# Replication: *Democratizing Optimization with Generative AI* (Simchi-Levi et al., SSRN 5511218)

Runs the full pipeline in Colab: tests → MILP demo → Qwen (via Ollama) benchmark → Figure 1.

**Before running:** `Runtime → Change runtime type → T4 GPU` (Qwen 7B runs ~10× faster than CPU).

## 1. Clone repo

In [1]:
REPO_URL = "https://github.com/<your-username>/genai-opt-replication.git"   # <- edit this

# Private repo? Add a GitHub token in Colab's Secrets panel (key icon) as GITHUB_TOKEN and set PRIVATE = True
PRIVATE = False
if PRIVATE:
    from google.colab import userdata
    REPO_URL = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
!rm -rf /content/repo && git clone -q $REPO_URL /content/repo
%cd /content/repo
!ls

/bin/bash: line 1: your-username: No such file or directory
[Errno 2] No such file or directory: '/content/repo'
/content
sample_data


## 2. Install Python deps + run tests

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q tests/

## 3. Install Ollama + pull Qwen

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen("nohup ollama serve > /content/ollama.log 2>&1 &", shell=True)
time.sleep(5)
MODEL = "qwen2.5:7b-instruct"     # also try qwen2.5:3b-instruct / qwen2.5:14b-instruct
!ollama pull $MODEL
!ollama run $MODEL "Reply with the word ok"

## 4. 4I demo (Insight, Interpretability, Interactivity, Improvisation)

In [ ]:
!python scripts/demo.py --llm --model $MODEL
from IPython.display import Markdown
Markdown(open("results/demo_transcript_llm.md").read())

## 5. Parser benchmark (rules vs. LLM)

In [ ]:
!python scripts/run_benchmark.py --backend rules
!python scripts/run_benchmark.py --backend llm --model $MODEL

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob("results/bench_*_summary.json")):
    s = json.load(open(f))
    for split in ("ID", "OOD"):
        rows.append({"backend": s["backend"], "split": split, **{k: s[split][k] for k in
                     ("n_questions", "question_acc", "set_acc", "e2e_acc", "median_parse_s")}})
pd.DataFrame(rows)

## 6. Figure 1 (Google Trends)
May hit HTTP 429 from Colab IPs; if so, download the CSV manually (see the script's docstring), upload it, and run with `--csv`.

In [ ]:
!python scripts/figure1_trends.py --fetch || echo "Trends blocked - use manual CSV"
import os
from IPython.display import Image, display
if os.path.exists("results/figure1_trends.png"):
    display(Image("results/figure1_trends.png"))

## 7. Save results

In [ ]:
!zip -qr /content/results.zip results data
from google.colab import files
files.download("/content/results.zip")